# ⚙️ Modul 01 Hands-on: The ML System & Data Pipeline

Pada hands-on ini, Anda akan mempraktikkan:
1. **The Working Blackbox**: Membangun pipeline klasifikasi tabular end-to-end dengan `sklearn.pipeline.Pipeline`.
2. **The Sabotage Lab: Data Leakage!** Membuktikan bahaya fitting scaler sebelum split.
3. **The Clean Build**: Menyusun pipeline modular dengan imputasi nilai hilang dan scaling yang benar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

print("Setup Scikit-Learn selesai!")

## 1. Menyiapkan Dataset Tabular Sintetis

In [ ]:
X, y = make_classification(n_samples=1000, n_features=10, n_informative=6, random_state=42)
print(f"Fitur X shape: {X.shape}, Label y shape: {y.shape}")

## 2. The Sabotage Lab: Data Leakage (Fitting Scaler Sebelum Split)
Mari kita buktikan apa dampaknya jika scaler di-fit ke SELURUH data.

In [ ]:
# SKENARIO 1: SABOTASE (Data Leakage)
scaler_leaked = StandardScaler()
X_leaked = scaler_leaked.fit_transform(X)
X_tr_leak, X_te_leak, y_tr, y_te = train_test_split(X_leaked, y, test_size=0.2, random_state=42)

clf_leak = LogisticRegression()
clf_leak.fit(X_tr_leak, y_tr)
acc_leak = accuracy_score(y_te, clf_leak.predict(X_te_leak))

# SKENARIO 2: BENAR (Pipeline Tanpa Leakage)
X_tr_clean, X_te_clean, y_tr_clean, y_te_clean = train_test_split(X, y, test_size=0.2, random_state=42)
clean_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression())
])
clean_pipeline.fit(X_tr_clean, y_tr_clean)
acc_clean = accuracy_score(y_te_clean, clean_pipeline.predict(X_te_clean))

print(f"Akurasi Skenario Bocor (Leakage): {acc_leak*100:.2f}%")
print(f"Akurasi Skenario Bersih (Pipeline): {acc_clean*100:.2f}%")
print("Kesimpulan: Pipeline memastikan data test tidak pernah bocor ke proses normalisasi!")

## 3. The Clean Build: Production-Grade Sklearn Pipeline

In [ ]:
y_pred = clean_pipeline.predict(X_te_clean)
print("Classification Report Produksi:")
print(classification_report(y_te_clean, y_pred))